# Scissor 入门教程：用 bulk 表型在单细胞中寻找相关细胞

参考教程：https://sunduanchen.github.io/Scissor/vignettes/Scissor_Tutorial.html

这份 Notebook 面向第一次接触 Scissor 的同学。目标不是把所有模型细节一次讲完，而是让你能完成一遍最小工作流：

1. 准备单细胞表达矩阵、bulk 表达矩阵和 bulk 表型。
2. 用 `Scissor()` 运行 Cox 或二分类模型。
3. 在 UMAP 上标出 Scissor+ / Scissor- 细胞。
4. 理解 `alpha`、`family`、`Scissor_pos`、`Scissor_neg` 的含义。

建议先完整跑一遍示例数据，再把 `location` 或本地文件路径换成自己的项目数据。

## 0. 环境准备

Scissor 是 R 包。Notebook 建议使用 R kernel。第一次运行前，确认已经安装：

- `Scissor`
- `Seurat`
- `Matrix`

如果服务器还没有安装 Scissor，可以在 R 中运行：

```r
install.packages("remotes")
remotes::install_github("sunduanchen/Scissor")
```

如果服务器无法联网，请让管理员提前把包装到对应 R 环境中。

In [1]:
library(Scissor)
library(Seurat)
library(Matrix)

set.seed(1234)

Loading required package: Seurat

Loading required package: Seurat

Loading required package: SeuratObject

Loading required package: SeuratObject

Loading required package: sp

Loading required package: sp


Attaching package: ‘SeuratObject’



Attaching package: ‘SeuratObject’


The following objects are masked from ‘package:base’:

    intersect, t


The following objects are masked from ‘package:base’:

    intersect, t


Loading required package: Matrix

Loading required package: Matrix

Warning message:
“package ‘Matrix’ was built under R version 4.3.2”
Warning message:
“package ‘Matrix’ was built under R version 4.3.2”


## 1. Scissor 需要哪三类输入？

Scissor 的核心问题是：给定一个 bulk 层面的表型，例如生存时间、突变状态、治疗响应，哪些单细胞更像是和这个表型相关？

因此它需要三类输入：

- `sc_dataset`：单细胞表达矩阵，行是基因，列是细胞；也可以是 Seurat 对象。
- `bulk_dataset`：bulk RNA-seq 表达矩阵，行是基因，列是 bulk 样本。
- `phenotype`：每个 bulk 样本的表型。可以是连续变量、二分类变量，或 Cox 生存信息。

最容易出错的是基因名和样本顺序：单细胞和 bulk 的基因名要有足够交集，`phenotype` 的顺序必须和 `bulk_dataset` 的列一致。

In [ ]:
# 官方教程示例数据位置
location <- "https://xialab.s3-us-west-2.amazonaws.com/Duanchen/Scissor_data/"

# LUAD 单细胞癌细胞表达数据。下载后会得到 raw count matrix：sc_dataset。
load(url(paste0(location, "scRNA-seq.RData")))

dim(sc_dataset)
sc_dataset[1:5, 1:5]

## 2. 快速检查并预处理单细胞数据

刚加载的 `sc_dataset` 是 raw count matrix，先看三个问题：

- 行名是不是基因名？
- 列名是不是细胞 barcode？
- 表达值是不是 raw counts 或非负表达量？

官方教程使用 `Seurat_preprocessing()` 把它转换成 Seurat 对象，并构建 Scissor 需要的细胞相似性网络。这个步骤会生成 PCA、t-SNE、UMAP 等降维结果。

In [ ]:
cat("基因数:", nrow(sc_dataset), "\n")
cat("细胞数:", ncol(sc_dataset), "\n")
cat("前 5 个基因:", paste(rownames(sc_dataset)[1:5], collapse = ", "), "\n")
cat("前 5 个细胞:", paste(colnames(sc_dataset)[1:5], collapse = ", "), "\n")

sc_dataset <- Seurat_preprocessing(sc_dataset, verbose = FALSE)
class(sc_dataset)
names(sc_dataset)

DimPlot(sc_dataset, reduction = "umap", label = TRUE, label.size = 5)

## 3. 示例 A：Cox 回归，寻找和生存相关的细胞

Cox 模型适用于生存数据。官方 LUAD 示例中，bulk 表型是每个患者的生存时间和结局状态。

在 `family = "cox"` 时：

- Scissor+ 通常解释为和较差生存相关的细胞。
- Scissor- 通常解释为和较好生存相关的细胞。

运行前先加载 bulk 表达和生存表型。

In [ ]:
load(url(paste0(location, "TCGA_LUAD_exp1.RData")))
load(url(paste0(location, "TCGA_LUAD_survival.RData")))

cat("bulk matrix:", nrow(bulk_dataset), "genes x", ncol(bulk_dataset), "samples\n")
head(bulk_survival)

stopifnot(all(colnames(bulk_dataset) == bulk_survival$TCGA_patient_barcode))
phenotype <- bulk_survival[, 2:3]
colnames(phenotype) <- c("time", "status")
head(phenotype)

# 检查基因交集。交集太少会影响结果可靠性。
common_genes <- intersect(rownames(sc_dataset), rownames(bulk_dataset))
cat("单细胞和 bulk 共有基因数:", length(common_genes), "\n")

In [ ]:
infos_cox <- Scissor(
  bulk_dataset,
  sc_dataset,
  phenotype,
  alpha = 0.05,
  family = "cox",
  Save_file = "Scissor_LUAD_survival.RData"
)

names(infos_cox)
length(infos_cox$Scissor_pos)
length(infos_cox$Scissor_neg)
infos_cox$para

## 4. 可视化 Scissor 结果

Scissor 返回的 `Scissor_pos` 和 `Scissor_neg` 是细胞名。为了画图，我们把它们写进 Seurat 对象的 metadata：

- `0`：未被 Scissor 选中。
- `1`：Scissor+。
- `2`：Scissor-。

如果你的单细胞对象还没有 UMAP，需要先用 Seurat 标准流程跑 PCA/UMAP；官方示例数据通常已经包含 UMAP。

In [ ]:
Scissor_select <- rep(0, ncol(sc_dataset))
names(Scissor_select) <- colnames(sc_dataset)
Scissor_select[infos_cox$Scissor_pos] <- 1
Scissor_select[infos_cox$Scissor_neg] <- 2

sc_dataset <- AddMetaData(
  sc_dataset,
  metadata = Scissor_select,
  col.name = "scissor_cox"
)

DimPlot(
  sc_dataset,
  reduction = "umap",
  group.by = "scissor_cox",
  cols = c("grey80", "indianred1", "royalblue"),
  pt.size = 1.0,
  order = c(2, 1)
)

## 5. 示例 B：二分类表型，TP53 突变 vs 野生型

当表型是 0/1 二分类时，使用 `family = "binomial"`。这里以 TCGA-LUAD 的 TP53 突变状态为例。

最重要的是确认编码方向：

- 如果 `TP53 mutant = 1`，`wild-type = 0`，Scissor+ 倾向于解释为更关联 TP53 突变。
- 如果你把 0/1 反过来，Scissor+ 和 Scissor- 的解释也会反过来。

所以建议显式设置 `tag`，让代码和结果说明更清楚。

In [ ]:
load(url(paste0(location, "TCGA_LUAD_exp2.RData")))
load(url(paste0(location, "TCGA_LUAD_TP53_mutation.RData")))

phenotype <- TP53_mutation
tag <- c("wild-type", "TP53 mutant")

table(phenotype)

In [ ]:
infos_tp53 <- Scissor(
  bulk_dataset,
  sc_dataset,
  phenotype,
  tag = tag,
  alpha = 0.5,
  family = "binomial",
  Save_file = "Scissor_LUAD_TP53_mutation.RData"
)

length(infos_tp53$Scissor_pos)
length(infos_tp53$Scissor_neg)
infos_tp53$para

In [ ]:
Scissor_select <- rep(0, ncol(sc_dataset))
names(Scissor_select) <- colnames(sc_dataset)
Scissor_select[infos_tp53$Scissor_pos] <- 1
Scissor_select[infos_tp53$Scissor_neg] <- 2

sc_dataset <- AddMetaData(
  sc_dataset,
  metadata = Scissor_select,
  col.name = "scissor_tp53"
)

DimPlot(
  sc_dataset,
  reduction = "umap",
  group.by = "scissor_tp53",
  cols = c("grey80", "indianred1", "royalblue"),
  pt.size = 1.0,
  order = c(2, 1)
)

## 6. 调 `alpha`：控制选中细胞比例

`alpha` 越大，模型通常越强调 L1 稀疏性，选中的细胞越少。入门阶段可以这样做：

1. 先用官方或文献推荐值跑一遍。
2. 再用 `Load_file` 复用前面保存的中间结果，尝试多个 `alpha`。
3. 不要只追求选中很多细胞；更建议记录不同 `alpha` 下选中比例是否稳定。

下面的代码使用前面保存的 Cox 文件，避免重复计算一部分耗时步骤。

In [ ]:
infos_alpha_scan <- Scissor(
  bulk_dataset,
  sc_dataset,
  phenotype,
  alpha = NULL,
  cutoff = 0.03,
  family = "cox",
  Load_file = "Scissor_LUAD_survival.RData"
)

## 7. 结果解释

- bulk 数据来源、样本数和表型定义。
- 单细胞数据来源、细胞数、是否只取某类细胞。
- `family`、`alpha`、`cutoff`。
- Scissor+ / Scissor- 细胞数和比例。
- 在 UMAP 上的位置，是否富集在某些已知细胞群或状态。
- 对二分类表型，明确 0/1 编码方向。

可以用下面的代码生成一个简短摘要。

In [ ]:
summarize_scissor <- function(info, total_cells, label = "Scissor result") {
  pos <- length(info$Scissor_pos)
  neg <- length(info$Scissor_neg)
  selected <- pos + neg
  data.frame(
    label = label,
    family = info$para$family,
    alpha = info$para$alpha,
    Scissor_pos = pos,
    Scissor_neg = neg,
    selected_percent = round(selected / total_cells * 100, 2)
  )
}

summarize_scissor(infos_tp53, ncol(sc_dataset), "TP53 mutation")

## 8. 换成自己数据时的检查清单

在自己的课题中使用 Scissor 前，逐项确认：

- bulk 和单细胞是否来自相近组织、物种和测序平台。
- bulk 表型是否可靠，样本量是否足够。
- bulk 表达矩阵列名和 phenotype 顺序是否一致。
- 基因 ID 是否统一，例如都用 gene symbol，或都用 Ensembl ID。
- Scissor 运行日志里的相关性五数概括是否合理；如果中位数接近 0，要谨慎解释。